# 📐 Image Orientation Classifier using ResNet18
# مصنف اتجاه الصور باستخدام ResNet18

يقوم هذا الكود بتدريب نموذج ResNet18 للتعرف على اتجاه الصور (0° أو 180°)
باستخدام نفس الصور التي دربنا عليها نموذج SVTR OCR

In [1]:
# Cell 1: المكتبات المطلوبة
import os
import random
import numpy as np
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

🔧 Using device: cpu


In [2]:
# Cell 2: الإعدادات
class Config:
    # Data paths - using the same data as SVTR training
    BASE_DIR = r"c:\Users\moom\vignette\lastocr"
    TRUE_DIR = os.path.join(BASE_DIR, "true")
    
    # Training settings
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    
    # Image settings - ResNet18 standard input
    IMG_SIZE = 224
    
    # Validation split
    VAL_SPLIT = 0.15
    
    # Model save path
    MODEL_PATH = os.path.join(BASE_DIR, "orientation_classifier_best.pth")

print("📁 Configuration:")
print(f"   Data directory: {Config.TRUE_DIR}")
print(f"   Batch size: {Config.BATCH_SIZE}")
print(f"   Epochs: {Config.EPOCHS}")
print(f"   Learning rate: {Config.LEARNING_RATE}")

📁 Configuration:
   Data directory: c:\Users\moom\vignette\lastocr/true
   Batch size: 32
   Epochs: 20
   Learning rate: 0.0001


In [3]:
# Cell 3: تحميل مسارات الصور
def load_image_paths():
    """Load all image paths from the true directory"""
    image_paths = []
    
    if os.path.exists(Config.TRUE_DIR):
        for filename in os.listdir(Config.TRUE_DIR):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(Config.TRUE_DIR, filename)
                image_paths.append(full_path)
    
    print(f"📷 Found {len(image_paths)} images in {Config.TRUE_DIR}")
    return image_paths

image_paths = load_image_paths()

📷 Found 0 images in c:\Users\moom\vignette\lastocr/true


In [4]:
# Cell 4: Dataset Class مع تدوير الصور
class OrientationDataset(Dataset):
    """
    Dataset for orientation classification.
    - Original images are labeled as 0 (correct orientation - 0°)
    - Rotated images (180°) are labeled as 1 (upside down)
    """
    
    def __init__(self, image_paths, transform=None, augment=True):
        self.image_paths = image_paths
        self.transform = transform
        self.augment = augment
        
        # Create samples: each image appears twice (original and rotated)
        self.samples = []
        for path in image_paths:
            self.samples.append((path, 0))  # Original = 0 degrees
            self.samples.append((path, 1))  # Rotated = 180 degrees
        
        # Shuffle samples
        random.shuffle(self.samples)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        
        # Load image
        try:
            image = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # Return a dummy sample
            image = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE), color='white')
            label = 0
        
        # Rotate if label is 1 (180 degrees)
        if label == 1:
            image = image.rotate(180)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

print("✅ OrientationDataset class defined")

✅ OrientationDataset class defined


In [5]:
# Cell 5: Transforms
# Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.0),  # Don't flip horizontally - affects orientation!
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Transforms defined")

✅ Transforms defined


In [6]:
# Cell 6: تقسيم البيانات وإنشاء DataLoaders
# Split image paths
train_paths, val_paths = train_test_split(
    image_paths, 
    test_size=Config.VAL_SPLIT, 
    random_state=SEED
)

print(f"📊 Dataset split:")
print(f"   Training images: {len(train_paths)} (x2 = {len(train_paths)*2} samples)")
print(f"   Validation images: {len(val_paths)} (x2 = {len(val_paths)*2} samples)")

# Create datasets
train_dataset = OrientationDataset(train_paths, transform=train_transform, augment=True)
val_dataset = OrientationDataset(val_paths, transform=val_transform, augment=False)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=Config.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    pin_memory=True
)

print(f"\n📦 DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")

ValueError: With n_samples=0, test_size=0.15 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
# Cell 7: عرض بعض العينات
def visualize_samples(dataset, n_samples=8):
    """Visualize some samples from the dataset"""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Get random samples
    indices = random.sample(range(len(dataset)), n_samples)
    
    for i, idx in enumerate(indices):
        image, label = dataset[idx]
        
        # Denormalize for visualization
        img = image.numpy().transpose(1, 2, 0)
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        orientation = "0° (Correct)" if label == 0 else "180° (Rotated)"
        color = 'green' if label == 0 else 'red'
        axes[i].set_title(f"Label: {label} - {orientation}", color=color, fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle('Sample Images from Training Dataset', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_samples(train_dataset)

In [ ]:
# Cell 8: بناء نموذج ResNet18
def create_model(pretrained=True):
    """
    Create ResNet18 model for binary classification (0° vs 180°)
    """
    # Load pretrained ResNet18
    model = models.resnet18(pretrained=pretrained)
    
    # Modify the final layer for binary classification
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 2)  # 2 classes: 0° and 180°
    )
    
    return model

# Create model
model = create_model(pretrained=True)
model = model.to(device)

print("🏗️ Model Architecture (final layers):")
print(model.fc)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 9: إعداد التدريب
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(), 
    lr=Config.LEARNING_RATE, 
    weight_decay=Config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    factor=0.5, 
    patience=3, 
    verbose=True
)

print("✅ Training setup complete:")
print(f"   Loss function: CrossEntropyLoss")
print(f"   Optimizer: AdamW")
print(f"   Scheduler: ReduceLROnPlateau")

In [ ]:
# Cell 10: دوال التدريب والتقييم
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100.*correct/total})
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating', leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✅ Training and validation functions defined")

In [ ]:
# Cell 11: حلقة التدريب الرئيسية
print("🚀 Starting Training...")
print("="*60)

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(Config.EPOCHS):
    print(f"\n📅 Epoch {epoch+1}/{Config.EPOCHS}")
    print("-"*40)
    
    # Training
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validation
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_acc)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print results
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, Config.MODEL_PATH)
        print(f"   💾 New best model saved! (Val Acc: {val_acc:.2f}%)")
    
    # Early stopping check
    current_lr = optimizer.param_groups[0]['lr']
    print(f"   Current LR: {current_lr:.2e}")

print("\n" + "="*60)
print(f"🏆 Training Complete! Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"📁 Best model saved to: {Config.MODEL_PATH}")

In [ ]:
# Cell 12: عرض نتائج التدريب
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(Config.BASE_DIR, 'orientation_training_curves.png'), dpi=150)
plt.show()
print("📊 Training curves saved!")

In [ ]:
# Cell 13: تحميل أفضل نموذج للاختبار
# Load best model
checkpoint = torch.load(Config.MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best model loaded from epoch {checkpoint['epoch']+1}")
print(f"   Best Val Accuracy: {checkpoint['val_acc']:.2f}%")

In [ ]:
# Cell 14: دالة التنبؤ للاستخدام مع نموذج OCR
class OrientationClassifier:
    """
    Orientation classifier to detect if an image is rotated 180 degrees.
    Use this before OCR to ensure correct text orientation.
    """
    
    def __init__(self, model_path, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        
        # Create model
        self.model = models.resnet18(pretrained=False)
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        print(f"✅ Orientation classifier loaded on {self.device}")
    
    def predict(self, image):
        """
        Predict the orientation of an image.
        
        Args:
            image: PIL Image or numpy array
            
        Returns:
            tuple: (is_rotated, confidence)
                - is_rotated: True if image needs 180° rotation
                - confidence: confidence score (0-1)
        """
        # Convert numpy to PIL if needed
        if isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        elif not isinstance(image, Image.Image):
            image = Image.open(image).convert('RGB')
        else:
            image = image.convert('RGB')
        
        # Transform
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_tensor)
            probs = torch.softmax(outputs, dim=1)
            pred = torch.argmax(probs, dim=1).item()
            confidence = probs[0, pred].item()
        
        is_rotated = (pred == 1)
        return is_rotated, confidence
    
    def correct_orientation(self, image):
        """
        Correct the orientation of an image if needed.
        
        Args:
            image: PIL Image or numpy array
            
        Returns:
            tuple: (corrected_image, was_rotated, confidence)
        """
        is_rotated, confidence = self.predict(image)
        
        if is_rotated:
            if isinstance(image, np.ndarray):
                corrected = cv2.rotate(image, cv2.ROTATE_180)
            else:
                corrected = image.rotate(180)
        else:
            corrected = image
        
        return corrected, is_rotated, confidence

print("✅ OrientationClassifier class defined")

In [ ]:
# Cell 15: اختبار المصنف
# Initialize classifier
classifier = OrientationClassifier(Config.MODEL_PATH)

# Test on some images
test_paths = random.sample(image_paths, min(6, len(image_paths)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, path in enumerate(test_paths):
    # Load image
    img = Image.open(path).convert('RGB')
    
    # Randomly rotate some images for testing
    actually_rotated = random.choice([True, False])
    if actually_rotated:
        img = img.rotate(180)
    
    # Predict
    is_rotated, confidence = classifier.predict(img)
    
    # Display
    axes[i].imshow(img)
    result = "180°" if is_rotated else "0°"
    actual = "180°" if actually_rotated else "0°"
    correct = (is_rotated == actually_rotated)
    color = 'green' if correct else 'red'
    axes[i].set_title(f"Predicted: {result} ({confidence:.2%})\nActual: {actual} | {'✓' if correct else '✗'}", 
                      color=color, fontsize=10)
    axes[i].axis('off')

plt.suptitle('Orientation Classifier Test Results', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: مثال على استخدام المصنف مع OCR
print("""
🔧 كيفية استخدام مصنف الاتجاه مع نموذج OCR:
================================================

from orientation_classifier import OrientationClassifier

# تحميل مصنف الاتجاه
orientation_clf = OrientationClassifier('orientation_classifier_best.pth')

# تحميل صورة
image = Image.open('your_image.jpg')

# تصحيح الاتجاه تلقائياً
corrected_image, was_rotated, confidence = orientation_clf.correct_orientation(image)

if was_rotated:
    print(f"تم تدوير الصورة 180° (ثقة: {confidence:.2%})")

# الآن استخدم الصورة المصححة مع OCR
ocr_result = your_ocr_model.predict(corrected_image)
""")

print(f"\n📁 Model saved to: {Config.MODEL_PATH}")
print(f"📊 Model size: {os.path.getsize(Config.MODEL_PATH) / 1024 / 1024:.2f} MB")

In [ ]:
# Cell 17: حفظ الكلاس كملف Python منفصل للاستخدام لاحقاً
classifier_code = '''#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Orientation Classifier for OCR Preprocessing
مصنف اتجاه الصور للمعالجة المسبقة لـ OCR

This module provides a ResNet18-based classifier to detect if an image
is upside down (rotated 180 degrees) and needs correction before OCR.
"""

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import cv2


class OrientationClassifier:
    """
    Orientation classifier to detect if an image is rotated 180 degrees.
    Use this before OCR to ensure correct text orientation.
    """
    
    def __init__(self, model_path, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        
        # Create model
        self.model = models.resnet18(pretrained=False)
        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        print(f"Orientation classifier loaded on {self.device}")
    
    def predict(self, image):
        """
        Predict the orientation of an image.
        
        Args:
            image: PIL Image, numpy array, or file path
            
        Returns:
            tuple: (is_rotated, confidence)
                - is_rotated: True if image needs 180° rotation
                - confidence: confidence score (0-1)
        """
        # Convert to PIL Image
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        elif isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:
            image = image.convert("RGB")
        
        # Transform and predict
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(input_tensor)
            probs = torch.softmax(outputs, dim=1)
            pred = torch.argmax(probs, dim=1).item()
            confidence = probs[0, pred].item()
        
        is_rotated = (pred == 1)
        return is_rotated, confidence
    
    def correct_orientation(self, image):
        """
        Correct the orientation of an image if needed.
        
        Args:
            image: PIL Image, numpy array, or file path
            
        Returns:
            tuple: (corrected_image, was_rotated, confidence)
        """
        # Store original type
        is_numpy = isinstance(image, np.ndarray)
        
        # Predict
        is_rotated, confidence = self.predict(image)
        
        # Correct if needed
        if is_rotated:
            if is_numpy:
                corrected = cv2.rotate(image, cv2.ROTATE_180)
            elif isinstance(image, str):
                corrected = Image.open(image).convert("RGB").rotate(180)
            else:
                corrected = image.rotate(180)
        else:
            if isinstance(image, str):
                corrected = Image.open(image).convert("RGB")
            else:
                corrected = image
        
        return corrected, is_rotated, confidence


if __name__ == "__main__":
    import sys
    
    if len(sys.argv) < 3:
        print("Usage: python orientation_classifier.py <model_path> <image_path>")
        sys.exit(1)
    
    model_path = sys.argv[1]
    image_path = sys.argv[2]
    
    classifier = OrientationClassifier(model_path)
    is_rotated, confidence = classifier.predict(image_path)
    
    print(f"Image: {image_path}")
    print(f"Orientation: {\"180°\" if is_rotated else \"0°\"}")
    print(f"Confidence: {confidence:.2%}")
'''

# Save the classifier module
module_path = os.path.join(Config.BASE_DIR, 'orientation_classifier.py')
with open(module_path, 'w', encoding='utf-8') as f:
    f.write(classifier_code)

print(f"✅ Classifier module saved to: {module_path}")

## ✅ تم الانتهاء!

### الملفات التي تم إنشاؤها:
1. **orientation_classifier_best.pth** - النموذج المدرب
2. **orientation_classifier.py** - كلاس Python للاستخدام
3. **orientation_training_curves.png** - منحنيات التدريب

### كيفية الاستخدام:
```python
from orientation_classifier import OrientationClassifier

# تحميل المصنف
clf = OrientationClassifier('orientation_classifier_best.pth')

# تصحيح اتجاه الصورة
corrected_img, was_rotated, confidence = clf.correct_orientation('image.jpg')
```